In [ ]:
%pip install openai python-dotenv

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from tenacity import retry, stop_after_attempt, wait_random_exponential

load_dotenv()

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ.get("GROQ_API_KEY")
)

MODEL="llama-3.1-8b-instant"

In [ ]:
""" 
Advanced retry logic using tenacity. this checks if there is a RateLimitError,
and auto retries using exponential backoff + jitter.
"""
@retry(
    reraise=True, # passes back final error
    stop=stop_after_attempt(5), # try 5 times before giving up
    wait=wait_random_exponential(min=1, max=10) # exponential backoff with random jitter
)

def fetch_groq_completion_with_backoff(messages_payload):
    """
    Executes a chat completion inside a jitter-backed retry loop 
    to withstand rate limits.
    """
    print("🔄 Attempting API request...")
    return client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages_payload
    )

def main():
    chat_history = [
        {"role": "user", "content": "Give me a quick tip on writing clean code."}
    ]

    try:
        print("--- Initiating Protected Request Loop ---")
        
        # call our function
        response = fetch_groq_completion_with_backoff(chat_history)
        
        print("\n--- Success Response Received ---")
        print(response.choices[0].message.content)
        
        # groq returs data directly as a payload object
        print("\n--- Usage Footprint Metadata ---")
        print(f"Tokens consumed this turn: {response.usage.total_tokens}")

    except groq.RateLimitError as e:
        # fallback block if even max retries are exhausted
        print(f"\n❌ High-level Exception: Rate limits completely exhausted.")
        print(f"Error Details: {e}")
    except Exception as e:
        print(f"\n❌ An unexpected error occurred: {e}")

if __name__ == "__main__":
    main()

--- Initiating Protected Request Loop ---
🔄 Attempting API request...

--- Success Response Received ---
Here's a quick tip on writing clean code:

**Extract and Refactor Small Functions**

If you find yourself looking at a function with multiple tasks or responsibilities, consider breaking it down into smaller, single-purpose functions.

This helps with:

* **Readability**: Easier to understand what each piece of code does
* **Maintainability**: Less chance of introducing errors when one function is updated
* **Testability**: Easier to test individual functions and ensure they work as expected

Example:
```python
# Bad
def calculate_invoice_total(invoice):
    subtotal = calculate_subtotal(invoice)
    tax = calculate_tax(subtotal)
    discount = calculate_discount(subtotal)
    return subtotal + tax - discount

# Good
def calculate_subtotal(invoice):
    # Calculate subtotal here

def calculate_tax(subtotal):
    # Calculate tax here

def calculate_discount(subtotal):
    # Calculate